# Data loading

MSGPACK to GeoParquet.

In [1]:
from ship_routing.app.routing import RoutingResult, RoutingLog
from load_tuning_results import (
    load_results_raw,
    load_result_for_key,
    load_results,
    get_journey_params_df,
    get_hyper_params_df,
    get_runtime_df,
    get_elite_df,
    get_forcing_df,
    get_diversity_df,
)

In [2]:
from pathlib import Path

import pandas as pd
import geopandas as gpd

In [3]:
import warnings

warnings.filterwarnings("ignore")

In [4]:
data_files = sorted(Path("../results/").glob("results_*.msgpack"))
print(len(data_files))
results = load_results(data_files)
len(results)

52


records:   0%|          | 0/177906 [00:00<?, ?it/s]

177906

In [5]:
# Load individual DataFrames
df_hyper = get_hyper_params_df(results)
df_journey = get_journey_params_df(results)
df_runtime = get_runtime_df(results)
df_elite = get_elite_df(results)
df_forcing = get_forcing_df(results)
df_diversity = get_diversity_df(results)

elite:   0%|          | 0/177906 [00:00<?, ?it/s]

diversity:   0%|          | 0/177906 [00:00<?, ?it/s]

diversity:   0%|          | 0/177906 [00:00<?, ?it/s]

In [6]:
# Merge: params (1-to-1) → runtime (1-to-1) → elite (1-to-many with left join)
df_merged = (
    df_hyper.merge(df_journey, left_index=True, right_index=True, how="inner")
    .merge(df_runtime, left_index=True, right_index=True, how="inner")
    .merge(df_elite, left_index=True, right_index=True, how="left")
    .merge(df_forcing, left_index=True, right_index=True, how="left")
    .merge(df_diversity, left_index=True, right_index=True, how="left")
)

df_merged

,hyper_population_size,hyper_random_seed,hyper_selection_acceptance_rate_warmup,hyper_mutation_width_fraction_warmup,hyper_mutation_displacement_fraction_warmup,hyper_generations,hyper_offspring_size,hyper_crossover_rounds,hyper_selection_quantile,hyper_selection_acceptance_rate,...,elite_length_relative,elite_cost_absolute,elite_cost_relative,geometry,forcing_scenario_name,forcing_currents_path,forcing_waves_path,forcing_winds_path,diversity_cost_q75_25_rms,diversity_cost_q50_00_rms
filename,,,,,,,,,,,,,,,,,,,,,
result:0000:seed3895092605,128,3895092605,0.3,0.99,0.10,2,4,1,0.10,0.00,...,1.044215,5.215584e+12,0.880062,LINESTRING (-11.000000000000002 50.00000000000...,baseline,data_large/cmems_mod_glo_phy_my_0.083deg_P1D-m...,data_large/cmems_mod_glo_wav_my_0.2deg_PT3H-i_...,data_large/cmems_obs-wind_glo_phy_my_l4_0.125d...,2.248120e+11,3.436609e+11
result:0000:seed3895092605,128,3895092605,0.3,0.99,0.10,2,4,1,0.10,0.00,...,1.044215,5.215584e+12,0.880062,LINESTRING (-11.000000000000002 50.00000000000...,baseline,data_large/cmems_mod_glo_phy_my_0.083deg_P1D-m...,data_large/cmems_mod_glo_wav_my_0.2deg_PT3H-i_...,data_large/cmems_obs-wind_glo_phy_my_l4_0.125d...,2.248120e+11,3.436609e+11
result:0001:seed1649728346,256,1649728346,0.3,0.99,0.10,4,4,2,0.25,0.00,...,1.105817,6.127407e+12,0.017000,LINESTRING (-10.999999999999998 50.00000000000...,baseline,data_large/cmems_mod_glo_phy_my_0.083deg_P1D-m...,data_large/cmems_mod_glo_wav_my_0.2deg_PT3H-i_...,data_large/cmems_obs-wind_glo_phy_my_l4_0.125d...,1.018314e+14,1.480951e+14
result:0001:seed1649728346,256,1649728346,0.3,0.99,0.10,4,4,2,0.25,0.00,...,1.105817,6.127407e+12,0.017000,LINESTRING (-10.999999999999998 50.00000000000...,baseline,data_large/cmems_mod_glo_phy_my_0.083deg_P1D-m...,data_large/cmems_mod_glo_wav_my_0.2deg_PT3H-i_...,data_large/cmems_obs-wind_glo_phy_my_l4_0.125d...,1.018314e+14,1.480951e+14
result:0002:seed3030206584,128,3030206584,0.3,0.99,0.25,2,4,1,0.10,0.25,...,1.035384,4.273094e+12,0.056787,"LINESTRING (-80.5 29.99999999999999, -79.65596...",baseline,data_large/cmems_mod_glo_phy_my_0.083deg_P1D-m...,data_large/cmems_mod_glo_wav_my_0.2deg_PT3H-i_...,data_large/cmems_obs-wind_glo_phy_my_l4_0.125d...,4.047755e+13,1.783403e+13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
result:3997:seed3352588423,64,3352588423,0.3,0.99,0.25,1,4,1,0.25,0.25,...,1.000004,1.417715e+13,0.995504,"LINESTRING (-80.5 30, -79.82039317830925 30.53...",no_winds,data_large/cmems_mod_glo_phy_my_0.083deg_P1D-m...,data_large/cmems_mod_glo_wav_my_0.2deg_PT3H-i_...,NaN,0.000000e+00,4.429708e+07
result:3998:seed2264517074,128,2264517074,0.3,0.99,0.10,1,4,0,0.10,0.00,...,1.002087,6.196180e+12,0.975562,LINESTRING (-11.000000000000002 50.00000000000...,no_winds,data_large/cmems_mod_glo_phy_my_0.083deg_P1D-m...,data_large/cmems_mod_glo_wav_my_0.2deg_PT3H-i_...,NaN,2.325568e+10,6.256184e+10
result:3998:seed2264517074,128,2264517074,0.3,0.99,0.10,1,4,0,0.10,0.00,...,1.002087,6.196180e+12,0.975562,LINESTRING (-11.000000000000002 50.00000000000...,no_winds,data_large/cmems_mod_glo_phy_my_0.083deg_P1D-m...,data_large/cmems_mod_glo_wav_my_0.2deg_PT3H-i_...,NaN,2.325568e+10,6.256184e+10


In [7]:
df_merged.columns

Index(['hyper_population_size', 'hyper_random_seed',
       'hyper_selection_acceptance_rate_warmup',
       'hyper_mutation_width_fraction_warmup',
       'hyper_mutation_displacement_fraction_warmup', 'hyper_generations',
       'hyper_offspring_size', 'hyper_crossover_rounds',
       'hyper_selection_quantile', 'hyper_selection_acceptance_rate',
       'hyper_mutation_width_fraction', 'hyper_mutation_displacement_fraction',
       'hyper_mutation_iterations', 'hyper_crossover_strategy',
       'hyper_hazard_penalty_multiplier', 'hyper_num_elites',
       'hyper_gd_iterations', 'hyper_learning_rate_time',
       'hyper_learning_rate_space', 'hyper_time_increment',
       'hyper_distance_increment', 'hyper_enable_adaptation',
       'hyper_target_relative_improvement', 'hyper_adaptation_scale_W',
       'hyper_adaptation_scale_D', 'hyper_W_min', 'hyper_W_max', 'hyper_D_min',
       'hyper_D_max', 'hyper_num_workers', 'hyper_executor_type',
       'journey_name', 'journey_lon_waypoints

In [8]:
gpd.GeoDataFrame(df_merged).to_parquet("../results/results_prelim.geoparquet")